## Flight delaying analytics

### Requirement

In [17]:
# Project Requirements:
# - Deliver an actionable dashboard using the framework of your choice (ex: MS Excel, Looker Studio, Jupyter Notebooks, Observable, etc...)
# - Use the publicly available "Marketing Carrier On-Time Performance" dataset from BTS (https://www.transtats.bts.gov/)
# - Report time period should span 7 years from January 2018 to January 2025
# - Flight delay is defined as more take-off or landing that is more than 15m difference from scheduled departure/arrival 
# - Provide reports that breakdown delays by Cause, Carrier, and Origin/Destination Airport/City 
# - Support zooming in to specific time ranges
# - Support aggregating results by week/month/quarter/year

### Code enviornment setup

In [18]:
import os
import requests
import zipfile
import io
import pandas as pd
import concurrent.futures
import glob
import urllib3
import time
import pyarrow

### Data Downloading 

In [19]:
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

REQUIRED_COLUMNS = [
    'Year', 'Quarter', 'Month', 'DayofMonth', 'FlightDate', # Time
    'Marketing_Airline_Network', 'Operating_Airline', # Carrier
     'OriginCityName',  'DestCityName',  # Location
     'DepDelayMinutes', 
     'ArrDelayMinutes', 
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay' # Reason 
]

TEMP_DIR = "bts_temp_parquets"
MASTER_PARQUET = "bts_delayed_flights_only_2018_2025.parquet"

def process_single_month(year, month):
    base_url = "https://transtats.bts.gov/PREZIP/On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_{}_{}.zip"
    url = base_url.format(year, month)
    temp_filename = os.path.join(TEMP_DIR, f"filtered_{year}_{month}.parquet")
    
    if os.path.exists(temp_filename):
        return f"Skipped: {year}-{month} already processed."
        
    for attempt in range(3):
        try:
            response = requests.get(url, stream=True, verify=False, timeout=60)
            response.raise_for_status()
            
            with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                csv_filename = z.namelist()[0] 
                
                with z.open(csv_filename) as f:
                    df = pd.read_csv(
                        f, 
                        usecols=lambda c: c.strip() in REQUIRED_COLUMNS, 
                        low_memory=False
                    )
                    
            df.rename(columns=lambda x: x.strip(), inplace=True)
            
            for col in REQUIRED_COLUMNS:
                if col not in df.columns:
                    df[col] = None
                    
            # Enforce strict types to prevent Parquet schema mismatch during merge
            numeric_cols = ['ArrDelayMinutes', 'DepDelayMinutes', 'Year', 'Month', 'DayofMonth']
            for col in numeric_cols:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    
            # Convert remaining columns to string to avoid mixed-type Arrow errors
            string_cols = [c for c in REQUIRED_COLUMNS if c not in numeric_cols]
            df[string_cols] = df[string_cols].astype(str)
            
            df = df[(df['ArrDelayMinutes'] >= 15.0) | (df['DepDelayMinutes'] >= 15.0)]
            df = df[REQUIRED_COLUMNS]
            
            # Save as Parquet instead of CSV
            df.to_parquet(temp_filename, engine='pyarrow', index=False)
            return f"Success: {year}-{month}"
            
        except Exception as e:
            if attempt == 2:
                return f"Failed {year}-{month} after 3 attempts: {e}"
            time.sleep(2) 

def run_fast_parquet_pipeline():
    os.makedirs(TEMP_DIR, exist_ok=True)
    tasks = []
    
    for year in range(2018, 2026):
        start_month = 1
        end_month = 12 if year < 2025 else 1
        for month in range(start_month, end_month + 1):
            tasks.append((year, month))
            
    print("  Phase 1: Multithreaded Download & Parquet Conversion  ")
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(process_single_month, y, m) for y, m in tasks]
        
        for future in concurrent.futures.as_completed(futures):
            print(future.result())
            
    print("\n  Phase 2: Merging into Master Parquet  ")
    all_temp_files = glob.glob(os.path.join(TEMP_DIR, "filtered_*.parquet"))
    
    if all_temp_files:
        print(f"Merging {len(all_temp_files)} Parquet files...")
        df_list = [pd.read_parquet(f, engine='pyarrow') for f in all_temp_files]
        master_df = pd.concat(df_list, ignore_index=True)
        
        master_df.sort_values(by=['Year', 'Month', 'DayofMonth'], inplace=True)
        
        # Save final Master Parquet
        master_df.to_parquet(MASTER_PARQUET, engine='pyarrow', index=False)
        print(f"Master Parquet created successfully with {len(master_df)} rows.")
        
        for f in all_temp_files:
            os.remove(f)
        os.rmdir(TEMP_DIR)
        print("Temporary files cleaned up.")
    else:
        print("No data found to merge.")

run_fast_parquet_pipeline()

--- Phase 1: Multithreaded Download & Parquet Conversion ---
Skipped: 2018-6 already processed.
Skipped: 2018-3 already processed.
Skipped: 2018-4 already processed.
Skipped: 2018-5 already processed.
Skipped: 2018-2 already processed.
Skipped: 2018-1 already processed.
Skipped: 2018-7 already processed.
Skipped: 2018-9 already processed.
Skipped: 2018-11 already processed.
Skipped: 2019-1 already processed.
Skipped: 2018-10 already processed.
Skipped: 2018-12 already processed.
Skipped: 2019-5 already processed.
Skipped: 2019-3 already processed.
Skipped: 2019-6 already processed.
Skipped: 2019-2 already processed.
Skipped: 2019-7 already processed.
Skipped: 2019-10 already processed.
Skipped: 2019-4 already processed.
Skipped: 2019-9 already processed.
Skipped: 2020-1 already processed.
Skipped: 2020-2 already processed.
Skipped: 2019-12 already processed.
Skipped: 2019-11 already processed.
Skipped: 2020-4 already processed.
Skipped: 2020-3 already processed.
Skipped: 2019-8 already

In [20]:
def patch_missing_month(year=2018, month=8):
    base_url = f"https://transtats.bts.gov/PREZIP/On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_{year}_{month}.zip"
    
    print(f"Downloading and fixing {year}-{month}...")
    response = requests.get(base_url, stream=True, verify=False)
    response.raise_for_status()
    
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        csv_filename = z.namelist()[0] 
        
        with z.open(csv_filename) as f:
            df_patch = pd.read_csv(
                f, 
                usecols=lambda c: c.strip() in REQUIRED_COLUMNS, 
                low_memory=False,
                encoding='latin1' 
            )
            
    df_patch.rename(columns=lambda x: x.strip(), inplace=True)
    
    for col in REQUIRED_COLUMNS:
        if col not in df_patch.columns:
            df_patch[col] = None
            
    numeric_cols = ['ArrDelayMinutes', 'DepDelayMinutes', 'Year', 'Month', 'DayofMonth']
    for col in numeric_cols:
        if col in df_patch.columns:
            df_patch[col] = pd.to_numeric(df_patch[col], errors='coerce')
            
    string_cols = [c for c in REQUIRED_COLUMNS if c not in numeric_cols]
    df_patch[string_cols] = df_patch[string_cols].astype(str)
    
    df_patch = df_patch[(df_patch['ArrDelayMinutes'] >= 15.0) | (df_patch['DepDelayMinutes'] >= 15.0)]
    df_patch = df_patch[REQUIRED_COLUMNS]
    
    print(f"Extracted {len(df_patch)} delayed flights for {year}-{month}.")
    
    df_master = pd.read_parquet(MASTER_PARQUET, engine='pyarrow')
    
    print("Appending fixed data to Master")
    df_combined = pd.concat([df_master, df_patch], ignore_index=True)
    
    print("Sorting chronologically")
    df_combined.sort_values(by=['Year', 'Month', 'DayofMonth'], inplace=True)
    
    print("Saving updated Master Parquet...")
    df_combined.to_parquet(MASTER_PARQUET, engine='pyarrow', index=False)
    
    print(f"Master Parquet updated. New row count: {len(df_combined)}")

patch_missing_month(2018, 8)

Extracted 181138 delayed flights for 2018-8.
Appending fixed data to Master
Sorting chronologically
Saving updated Master Parquet...
Master Parquet updated. New row count: 10912791


### Verifying 


In [21]:
df = pd.read_parquet(MASTER_PARQUET, engine='pyarrow')

print("\n  1. DataFrame Shape  ")
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")

print("\n  2. Data Types & Memory Usage  ")
df.info(show_counts=True)

print("\n  3. Missing Values Count  ")
missing_stats = df.isna().sum()
print(missing_stats[missing_stats > 0])

print("\n  4. Basic Descriptive Statistics (Delay Minutes)  ")
delay_cols = ['DepDelayMinutes', 'ArrDelayMinutes']
display(df[delay_cols].describe().round(2))


  1. DataFrame Shape  
Total Rows: 10,912,791
Total Columns: 16

  2. Data Types & Memory Usage  
<class 'pandas.DataFrame'>
RangeIndex: 10912791 entries, 0 to 10912790
Data columns (total 16 columns):
 #   Column                     Non-Null Count     Dtype  
---  ------                     --------------     -----  
 0   Year                       10912791 non-null  int64  
 1   Quarter                    10912791 non-null  str    
 2   Month                      10912791 non-null  int64  
 3   DayofMonth                 10912791 non-null  int64  
 4   FlightDate                 10912791 non-null  str    
 5   Marketing_Airline_Network  10912791 non-null  str    
 6   Operating_Airline          10912791 non-null  str    
 7   OriginCityName             10912791 non-null  str    
 8   DestCityName               10912791 non-null  str    
 9   DepDelayMinutes            10912747 non-null  float64
 10  ArrDelayMinutes            10859584 non-null  float64
 11  CarrierDelay             

,DepDelayMinutes,ArrDelayMinutes
count,10912747.00,10859584.00
mean,57.90,58.34
std,91.37,90.32
min,0.00,0.00
25%,17.00,18.00
50%,32.00,32.00
75%,67.00,67.00
max,7223.00,7232.00


In [22]:

def verify_project_requirements():
    df = pd.read_parquet(MASTER_PARQUET, engine='pyarrow')
    
    print("  1. Time Period (Jan 2018 to Jan 2025)  ")
    min_year = df['Year'].min()
    max_year = df['Year'].max()
    print(f"Data spans from Year {min_year} to Year {max_year}.")
    if min_year == 2018 and max_year == 2025:
        print("[PASS] Time period requirement met.\n")
    else:
        print("[FAIL] Time period does not match requirements.\n")

    print("  2. Delay Definition (> 15m)  ")
    min_arr_delay = df['ArrDelayMinutes'].min()
    min_dep_delay = df['DepDelayMinutes'].min()
    
    # We used an OR logic. So the absolute minimum delay across the whole dataset 
    # must be >= 15 in at least ONE of the columns for every single row.
    invalid_rows = df[(df['ArrDelayMinutes'] < 15.0) & (df['DepDelayMinutes'] < 15.0)]
    if len(invalid_rows) == 0:
        print(f"[PASS] 100% of rows contain a delay >= 15 mins.\n")
    else:
        print(f"[FAIL] Found {len(invalid_rows)} rows with delays under 15 mins.\n")

    print("  3. Required Breakdown Dimensions  ")
    required_dims = {
        'Carrier': 'Marketing_Airline_Network', 
        'Origin City': 'OriginCityName', 
        'Dest City': 'DestCityName'
    }
    
    for human_name, col_name in required_dims.items():
        missing_pct = (df[col_name].isna().sum() / len(df)) * 100
        unique_vals = df[col_name].nunique()
        print(f"{human_name} Column ('{col_name}'):")
        print(f"  -> {unique_vals} unique values found.")
        print(f"  -> {missing_pct:.2f}% null values.")
        if missing_pct > 5.0:
            print(f"  -> [WARNING] High null rate in {human_name}!\n")
        else:
            print(f"  -> [PASS] Dimension is highly populated.\n")

    print("  4. Delay Causes  ")
    causes = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
    valid_causes_found = False
    
    for cause in causes:
        valid_count = df[cause].notna().sum()
        if valid_count > 0:
            valid_causes_found = True
            
    if valid_causes_found:
        print("[PASS] Delay cause data successfully populated for severe delays.\n")
    else:
        print("[FAIL] All delay cause columns are completely empty.\n")
        
    print("  5. Carriers Check  ")
    print(df['Marketing_Airline_Network'].value_counts().head(5))

verify_project_requirements()

  1. Time Period (Jan 2018 to Jan 2025)  
Data spans from Year 2018 to Year 2025.
[PASS] Time period requirement met.

  2. Delay Definition (> 15m)  
[PASS] 100% of rows contain a delay >= 15 mins.

  3. Required Breakdown Dimensions  
Carrier Column ('Marketing_Airline_Network'):
  -> 11 unique values found.
  -> 0.00% null values.
  -> [PASS] Dimension is highly populated.

Origin City Column ('OriginCityName'):
  -> 383 unique values found.
  -> 0.00% null values.
  -> [PASS] Dimension is highly populated.

Dest City Column ('DestCityName'):
  -> 383 unique values found.
  -> 0.00% null values.
  -> [PASS] Dimension is highly populated.

  4. Delay Causes  
[PASS] Delay cause data successfully populated for severe delays.

  5. Carriers Check  
Marketing_Airline_Network
AA    2773474
WN    2230243
UA    1988518
DL    1844518
B6     541851
Name: count, dtype: int64
